# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Pipeline 1:** Bi-Encoder -> Top 10 (cosine) -> Top 3 -> LLM -> Answer 1  
**Pipeline 2:** same Top 10 -> Cross-Encoder re-rank -> Top 3 -> LLM (same prompt) -> Answer 2

Runs on a Kaggle GPU (T4). Everything is open-weights, so no API key is needed.

**Before running:** turn on *Accelerator: GPU T4 x2* and *Internet: On* (the PDF is downloaded in the next cells).

## 0. Install & imports

In [1]:
!pip -q install sentence-transformers faiss-cpu pypdf accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.5 MB/s eta 0:00:00


### Download the PDF (Catan Game Rules & Almanac, 5th edition, ~16 pages). Needs Kaggle Internet: On.

In [2]:
!wget -q -O /kaggle/working/catan_rules.pdf "https://world-of-board-games.com.sg/docs/Catan-5ed.pdf"
!ls -lh /kaggle/working/catan_rules.pdf

-rw-r--r-- 1 root root 3.5M Aug 16  2020 /kaggle/working/catan_rules.pdf


In [3]:
!curl -sI https://huggingface.co | head -n 1

In [4]:
import re, time, statistics, warnings
import numpy as np
import pandas as pd
import torch, faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


## 1. Config

In [5]:
# ---------- EDIT THESE ----------
PDF_PATH   = "/kaggle/working/catan_rules.pdf"

EMB_MODEL  = "BAAI/bge-small-en-v1.5"                    # Bi-Encoder
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge-specific; set "" for other models
CE_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"      # Cross-Encoder (note: 'ms-marco' with a hyphen)
LLM_MODEL  = "Qwen/Qwen2.5-3B-Instruct"                  # open-weights LLM, fits a T4 in fp16

CHUNK_WORDS   = 120    # words per chunk
CHUNK_OVERLAP = 30     # overlapping words between neighbouring chunks
TOP_K_RETRIEVE = 10    # candidates from the bi-encoder
TOP_K_CONTEXT  = 3     # chunks given to the LLM
MAX_NEW_TOKENS = 300
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 2. Load PDF and chunk it (one corpus shared by both pipelines)

In [6]:
reader = PdfReader(PDF_PATH)
words, word_pages = [], []
for pno, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    text = re.sub(r"\s+", " ", text).strip()
    for w in text.split(" "):
        if w:
            words.append(w)
            word_pages.append(pno)

chunks = []   # list of dicts: id, page, text
step = CHUNK_WORDS - CHUNK_OVERLAP
for start in range(0, len(words), step):
    piece = words[start:start + CHUNK_WORDS]
    if len(piece) < 30:      # drop tiny tail
        continue
    chunks.append({"id": len(chunks), "page": word_pages[start], "text": " ".join(piece)})

print(f"Pages: {len(reader.pages)} | Words: {len(words)} | Chunks: {len(chunks)}")
print("\nSample chunk:\n", chunks[0]["text"][:400])

Pages: 16 | Words: 10127 | Chunks: 113

Sample chunk:
 Klaus Teuber Game RulesGame Rules && almanac almanac 22 Dear Settlers, To make it as easy as possible for you to start playing Catan®, we use an award-winning rules system, which consists of 3 parts. First, if you do not know how to play Catan, please read the Game Overview on page 16 (the back cover). Next, read the Game Rules on pages 2-6 (red borders) and start to play the game. If you have que


## 3. Pipeline 1 setup: dense index (Bi-Encoder + FAISS inner product on normalised vectors = cosine)

In [7]:
bi_encoder = SentenceTransformer(EMB_MODEL, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode([c["text"] for c in chunks], batch_size=64,
                              normalize_embeddings=True, show_progress_bar=True).astype("float32")
print(f"Indexing time: {time.perf_counter()-t0:.2f}s")

index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)
print("Index size:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Indexing time: 1.37s
Index size: 113


## 4. Pipeline 2 setup: Cross-Encoder

In [8]:
cross_encoder = CrossEncoder(CE_MODEL, device=DEVICE, max_length=512)
print("Cross-encoder loaded")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded


## 5. LLM (same model and same prompt template for both answers)

In [9]:
tok = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map="auto")
llm.eval()

SYSTEM_PROMPT = (
    "You are a careful assistant answering questions about a document. "
    "Use ONLY the provided context. If the context does not contain the answer, say "
    "'The provided context does not contain the answer.' Mention any exceptions or restrictions found in the context."
)

def build_prompt(query, ctx_chunks):
    ctx = "\n\n".join(f"[Chunk {i+1} | page {c['page']}]\n{c['text']}" for i, c in enumerate(ctx_chunks))
    user = f"Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer:"
    return tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_answer(query, ctx_chunks):
    prompt = build_prompt(query, ctx_chunks)
    inputs = tok(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 6. Retrieval functions with timing

In [10]:
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Stage 1: bi-encoder top-k. Returns (ids, scores, seconds)."""
    sync(); t0 = time.perf_counter()
    q_emb = bi_encoder.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, k)
    sync(); dt = time.perf_counter() - t0
    return ids[0].tolist(), scores[0].tolist(), dt

def rerank(query, cand_ids):
    """Stage 2: cross-encoder scores all (query, chunk) pairs, sorts descending. Returns (ids, scores, seconds)."""
    sync(); t0 = time.perf_counter()
    pairs = [(query, chunks[i]["text"]) for i in cand_ids]
    ce_scores = cross_encoder.predict(pairs, batch_size=10)
    order = np.argsort(-ce_scores)
    sync(); dt = time.perf_counter() - t0
    return [cand_ids[i] for i in order], [float(ce_scores[i]) for i in order], dt

# warm-up so first-call overhead doesn't pollute latency numbers
_ids, _, _ = baseline_retrieve("warm up")
_ = rerank("warm up", _ids)
print("Warm-up done")

Warm-up done


## 7. Test queries

Fill this list with **your own** queries after reading your PDF. For each one:
- `category`: Keyword Trap / Exception / Negation / Buried chunk / Normal
- `gold_phrase`: a short, distinctive phrase (lowercase) that appears in the chunk that truly answers the question. It is used to track where that chunk ranks before and after re-ranking.

The queries below are pre-written for the Catan rulebook. Run them, then keep the ones that show a rank shift (gold chunk outside Top 3 in baseline, inside Top 3 after re-ranking) and drop or replace the rest.

In [11]:
QUERIES = [
    {"category": "Keyword Trap",  "query": "How far apart must two houses be built from each other?", "gold_phrase": "adjacent intersections are vacant"},
    {"category": "Keyword Trap",  "query": "I am holding nine cards and someone rolls a seven. How many do I lose?", "gold_phrase": "if you have 9 resource cards"},
    {"category": "Exception",     "query": "I just bought a development card this turn. Is there any case where I can use it right away?", "gold_phrase": "immediately reveal this card"},
    {"category": "Exception",     "query": "An opponent cut my longest road with a settlement and now I am tied. Do I lose the Longest Road card?", "gold_phrase": "you are tied for longest road"},
    {"category": "Exception",     "query": "Can I play more than one development card in a single turn?", "gold_phrase": "any number of victory point cards"},
    {"category": "Negation",      "query": "Is it allowed to trade 2 wool for 1 wool?", "gold_phrase": "may not trade like resources"},
    {"category": "Negation",      "query": "After rolling a seven, can I just leave the robber where it is?", "gold_phrase": "may not choose to leave the robber"},
    {"category": "Negation",      "query": "Can I build a city directly on an empty intersection?", "gold_phrase": "cannot build a city directly"},
    {"category": "Buried chunk",  "query": "I own a settlement on the ore harbor. Can I also trade wool at 2 to 1?", "gold_phrase": "any other resource type at a more favorable rate"},
    {"category": "Buried chunk",  "query": "I reach 10 victory points during another player's turn. Do I win immediately?", "gold_phrase": "wait until your next turn"},
]

In [12]:
# Sanity check: every gold_phrase must exist in at least one chunk (PDF text extraction can differ slightly)
for item in QUERIES:
    hits = [c["id"] for c in chunks if item["gold_phrase"].lower() in c["text"].lower()]
    print("OK  " if hits else "MISS", hits, "|", item["gold_phrase"])

OK   [19] | adjacent intersections are vacant
OK   [68] | if you have 9 resource cards
OK   [36, 37] | immediately reveal this card
OK   [51, 52] | you are tied for longest road
OK   [26] | any number of victory point cards
OK   [85] | may not trade like resources
OK   [70] | may not choose to leave the robber
OK   [31] | cannot build a city directly
OK   [55] | any other resource type at a more favorable rate
OK   [40] | wait until your next turn


## 8. Run both pipelines on every query

In [13]:
def find_gold_ids(phrase):
    p = phrase.lower()
    return [c["id"] for c in chunks if p in c["text"].lower()]

def rank_of(gold_ids, ranked_ids):
    """1-based rank of the best-placed gold chunk, or None if absent from list."""
    ranks = [ranked_ids.index(g) + 1 for g in gold_ids if g in ranked_ids]
    return min(ranks) if ranks else None

N_REPEATS = 5   # latency is the median of several runs
results = []

for item in QUERIES:
    q = item["query"]
    gold_ids = find_gold_ids(item["gold_phrase"])
    if not gold_ids:
        print(f"[WARN] gold_phrase not found in any chunk for: {q!r}")

    # --- latency (median over repeats) ---
    t_base, t_rerank = [], []
    for _ in range(N_REPEATS):
        ids10, sc10, tb = baseline_retrieve(q)
        r_ids, r_sc, tr = rerank(q, ids10)
        t_base.append(tb); t_rerank.append(tr)

    base_top3 = ids10[:TOP_K_CONTEXT]
    rr_top3 = r_ids[:TOP_K_CONTEXT]

    # --- answers ---
    ans1 = generate_answer(q, [chunks[i] for i in base_top3])
    ans2 = generate_answer(q, [chunks[i] for i in rr_top3])

    results.append(dict(
        category=item["category"], query=q, gold_ids=gold_ids,
        base_ids=ids10, base_scores=sc10, rr_ids=r_ids, rr_scores=r_sc,
        gold_rank_base=rank_of(gold_ids, ids10), gold_rank_rr=rank_of(gold_ids, r_ids),
        t_base=statistics.median(t_base), t_rerank=statistics.median(t_rerank),
        ans1=ans1, ans2=ans2))
print("Done")

Done


## 9. Rank-shift table

In [14]:
rows = []
for r in results:
    b, a = r["gold_rank_base"], r["gold_rank_rr"]
    rows.append({
        "Category": r["category"],
        "Query": r["query"][:70],
        "Gold chunk ids": r["gold_ids"],
        "Rank (baseline)": b if b else ">10",
        "Rank (re-ranked)": a if a else ">10",
        "In Top3 baseline": b is not None and b <= TOP_K_CONTEXT,
        "In Top3 re-ranked": a is not None and a <= TOP_K_CONTEXT,
    })
rank_df = pd.DataFrame(rows)
rank_df

,Category,Query,Gold chunk ids,Rank (baseline),Rank (re-ranked),In Top3 baseline,In Top3 re-ranked
0,Keyword Trap,How far apart must two houses be built from each other?,[19],3,2,True,True
1,Keyword Trap,I am holding nine cards and someone rolls a seven. How many do I lose?,[68],2,1,True,True
2,Exception,I just bought a development card this turn. Is there any case where I,"[36, 37]",1,1,True,True
3,Exception,An opponent cut my longest road with a settlement and now I am tied. D,"[51, 52]",2,1,True,True
4,Exception,Can I play more than one development card in a single turn?,[26],>10,>10,False,False
5,Negation,Is it allowed to trade 2 wool for 1 wool?,[85],>10,>10,False,False
6,Negation,"After rolling a seven, can I just leave the robber where it is?",[70],>10,>10,False,False
7,Negation,Can I build a city directly on an empty intersection?,[31],5,1,False,True
8,Buried chunk,I own a settlement on the ore harbor. Can I also trade wool at 2 to 1?,[55],1,1,True,True
9,Buried chunk,I reach 10 victory points during another player's turn. Do I win immed,[40],2,1,True,True


## 10. Detailed view per query: top-10 order, promoted chunks, Answer 1 vs Answer 2

In [15]:
for r in results:
    print("=" * 110)
    print(f"[{r['category']}] {r['query']}")
    print(f"Gold chunk(s): {r['gold_ids']} | rank baseline: {r['gold_rank_base']} -> re-ranked: {r['gold_rank_rr']}")
    print("\nBaseline order (chunk id : cosine):")
    print("  ", [(i, round(s, 3)) for i, s in zip(r["base_ids"], r["base_scores"])])
    print("Re-ranked order (chunk id : CE score):")
    print("  ", [(i, round(s, 2)) for i, s in zip(r["rr_ids"], r["rr_scores"])])

    promoted = [i for i in r["rr_ids"][:TOP_K_CONTEXT] if r["base_ids"].index(i) + 1 > TOP_K_CONTEXT]
    for i in promoted:
        print(f"\n  >> Chunk {i} PROMOTED: baseline rank #{r['base_ids'].index(i)+1} -> re-ranked rank #{r['rr_ids'].index(i)+1} (page {chunks[i]['page']})")
        print("     ", chunks[i]["text"][:350], "...")

    print("\n--- Answer 1 (baseline) ---\n", r["ans1"])
    print("\n--- Answer 2 (re-ranked) ---\n", r["ans2"])
print("=" * 110)

[Keyword Trap] How far apart must two houses be built from each other?
Gold chunk(s): [19] | rank baseline: 3 -> re-ranked: 2

Baseline order (chunk id : cosine):
   [(75, 0.664), (76, 0.646), (19, 0.625), (71, 0.621), (18, 0.609), (11, 0.601), (17, 0.599), (74, 0.59), (108, 0.587), (82, 0.584)]
Re-ranked order (chunk id : CE score):
   [(75, -7.62), (19, -7.84), (71, -7.93), (108, -9.35), (76, -9.85), (18, -10.09), (74, -10.13), (17, -10.35), (82, -10.64), (11, -11.21)]

  >> Chunk 71 PROMOTED: baseline rank #4 -> re-ranked rank #3 (page 11)
      the resource production of each terrain hex adjacent to your settlements. You must meet 2 conditions when building a settlement: (1) Y our settlement must always connect to 1 or more of your own roads Y. (2) Y ou must observe the Distance Rule Y. For an example of the distance rule, see Illustration M on page 12. Example: See Illustration L. It is  ...

--- Answer 1 (baseline) ---
 The provided context does not contain the answer.

--- Answe

## 11. Latency profiling

In [16]:
lat_rows = []
for r in results:
    lat_rows.append({
        "Query": r["query"][:60],
        "Baseline retrieval (ms)": round(r["t_base"] * 1000, 2),
        "Re-ranking of 10 (ms)": round(r["t_rerank"] * 1000, 2),
        "Pipeline 2 total (ms)": round((r["t_base"] + r["t_rerank"]) * 1000, 2),
    })
lat_df = pd.DataFrame(lat_rows)
avg = lat_df.drop(columns="Query").mean().round(2)
lat_df.loc[len(lat_df)] = ["AVERAGE"] + avg.tolist()
lat_df

,Query,Baseline retrieval (ms),Re-ranking of 10 (ms),Pipeline 2 total (ms)
0,How far apart must two houses be built from each other?,10.25,30.45,40.70
1,I am holding nine cards and someone rolls a seven. How many,10.48,29.19,39.67
2,I just bought a development card this turn. Is there any cas,10.52,22.49,33.01
3,An opponent cut my longest road with a settlement and now I,10.95,20.69,31.64
4,Can I play more than one development card in a single turn?,10.86,25.13,35.99
5,Is it allowed to trade 2 wool for 1 wool?,10.35,28.39,38.75
6,"After rolling a seven, can I just leave the robber where it",10.45,23.41,33.86
7,Can I build a city directly on an empty intersection?,10.41,19.53,29.94
8,I own a settlement on the ore harbor. Can I also trade wool,11.55,33.15,44.70
9,I reach 10 victory points during another player's turn. Do I,10.37,23.07,33.44


### Scaling experiment (supports the "Scaling Question")
Cross-encoder cost grows linearly with the number of (query, chunk) pairs, while the bi-encoder only embeds the query once (chunks are pre-embedded) and searches the index.

In [17]:
q = QUERIES[0]["query"]
sizes = [10, 50, 100, 200, min(400, len(chunks))]
sizes = sorted(set(s for s in sizes if s <= len(chunks)))
scale_rows = []
for n in sizes:
    ids = list(range(n))
    times = []
    for _ in range(3):
        _, _, dt = rerank(q, ids)
        times.append(dt)
    scale_rows.append({"Chunks scored by cross-encoder": n, "Time (ms)": round(statistics.median(times) * 1000, 1)})
scale_df = pd.DataFrame(scale_rows)
scale_df["ms per chunk"] = (scale_df["Time (ms)"] / scale_df["Chunks scored by cross-encoder"]).round(2)
display(scale_df)

ms_per_chunk = scale_df["ms per chunk"].iloc[-1]
for N in [len(chunks), 10_000, 100_000, 1_000_000]:
    print(f"Estimated cross-encoder time for {N:>9,} chunks: {ms_per_chunk * N / 1000:,.1f} s per query")

,Chunks scored by cross-encoder,Time (ms),ms per chunk
0,10,28.9,2.89
1,50,97.4,1.95
2,100,203.7,2.04
3,113,237.0,2.10


Estimated cross-encoder time for       113 chunks: 0.2 s per query
Estimated cross-encoder time for    10,000 chunks: 21.0 s per query
Estimated cross-encoder time for   100,000 chunks: 210.0 s per query
Estimated cross-encoder time for 1,000,000 chunks: 2,100.0 s per query


## 12. Notes for your report (write the final text in your own words)

The assignment requires a report that is not AI-generated, so this notebook only produces the **evidence**. Use these outputs:

1. **Architecture comparison**: your own explanation (no code needed).
2. **Rank shift + answer comparison**: Section 9 table, plus Section 10 for Answer 1 vs Answer 2.
3. **Latency**: Section 11 table.
4. **Scaling question**: Section 11 scaling experiment.